# 🧠 nanoGPT: Ein GPT-Modell von Grund auf trainieren

**Lernziel:** Verstehe die GPT-Architektur, trainiere ein eigenes Sprachmodell und generiere Text.

Basierend auf [Andrej Karpathys nanoGPT](https://github.com/karpathy/nanoGPT) — der kleinstmöglichen, verständlichen GPT-Implementierung.

## Was du in diesem Notebook lernst:
1. **GPT-Architektur** — Token Embeddings, Self-Attention, Transformer-Blöcke
2. **Datenvorbereitung** — Tokenisierung, Batching
3. **Training** — Loss, Optimizer, Learning Rate Schedule
4. **Textgenerierung** — Sampling-Strategien (Temperature, Top-k, Top-p)
5. **Attention-Visualisierung** — Was „sieht" das Modell?
6. **Scaling Laws** — Wie skaliert Performance mit Modellgröße?

## 1. Umgebung & Imports

In [ ]:
import os
import sys
import math
import time
from collections import OrderedDict
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import matplotlib.pyplot as plt

# nanoGPT-Pfad hinzufügen
sys.path.insert(0, '/opt/data/nanoGPT')

from model import GPT, GPTConfig, LayerNorm, CausalSelfAttention, Block
from config import get_train_config

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🔥 Gerät: {device}')
print(f'🔥 PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')
    print(f'🔥 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Die GPT-Architektur verstehen

GPT (Generative Pre-trained Transformer) besteht aus diesen Kernkomponenten:

```
Input-Text → Token-Embedding + Positions-Embedding
           → [Transformer-Block] × N (Attention + MLP)
           → LayerNorm → Linear → Softmax → Nächstes Token
```

### 2.1 GPTConfig — Die Architektur-Parameter

In [ ]:
# GPT-2 Modellfamilie — von klein bis groß
model_sizes = {
    'GPT-2 Baby (Debug)':  dict(n_layer=6,  n_head=6,  n_embd=384,  block_size=256),
    'GPT-2 Small (124M)':  dict(n_layer=12, n_head=12, n_embd=768,  block_size=1024),
    'GPT-2 Medium (350M)': dict(n_layer=24, n_head=16, n_embd=1024, block_size=1024),
    'GPT-2 Large (774M)':  dict(n_layer=36, n_head=20, n_embd=1280, block_size=1024),
    'GPT-2 XL (1.5B)':     dict(n_layer=48, n_head=25, n_embd=1600, block_size=1024),
}

for name, cfg in model_sizes.items():
    config = GPTConfig(**cfg)
    model = GPT(config)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'{name:25s}: {n_params:6.1f}M Parameter | d_model={cfg["n_embd"]:4d} | Layers={cfg["n_layer"]:2d} | Heads={cfg["n_head"]:2d}')

### 2.2 Parameter-Anzahl berechnen

Woher kommen die 124M Parameter von GPT-2 Small? Lass uns das nachrechnen:

In [ ]:
def count_parameters(n_layer=12, n_head=12, n_embd=768, block_size=1024, vocab_size=50257):
    """Berechnet die Parameter-Anzahl eines GPT-Modells."""
    out = OrderedDict()
    
    # Token + Position Embeddings
    out['Token-Embedding'] = n_embd * vocab_size
    out['Positions-Embedding'] = n_embd * block_size
    
    # Ein Attention-Block
    out['Attention/LayerNorm'] = n_embd
    out['Attention/QKV'] = n_embd * 3 * n_embd  # Q, K, V in einer Matrix
    out['Attention/Proj'] = n_embd * n_embd
    attn_total = out['Attention/LayerNorm'] + out['Attention/QKV'] + out['Attention/Proj']
    
    # Ein MLP-Block (Feed-Forward)
    ffw_size = 4 * n_embd
    out['MLP/LayerNorm'] = n_embd
    out['MLP/FFW'] = n_embd * ffw_size
    out['MLP/Proj'] = ffw_size * n_embd
    mlp_total = out['MLP/LayerNorm'] + out['MLP/FFW'] + out['MLP/Proj']
    
    # Gesamt
    block_total = attn_total + mlp_total
    out['Pro Block'] = block_total
    out['Alle Blöcke'] = n_layer * block_total
    out['Final LayerNorm'] = n_embd
    out['Output (LM Head)'] = 0  # Weight Tying: teilt Gewichte mit Token-Embedding
    
    total = out['Token-Embedding'] + out['Positions-Embedding'] + out['Alle Blöcke'] + out['Final LayerNorm']
    out['GESAMT'] = total
    
    return out

params = count_parameters()
print(f"{'Komponente':25s} {'Parameter':>12s}  {'Anteil':>8s}")
print("-" * 50)
for name, count in params.items():
    pct = count / params['GESAMT'] * 100 if params['GESAMT'] > 0 else 0
    print(f"{name:25s} {count:>12,d}  {pct:>7.1f}%")

print(f"\n✅ Erwartet: 124,337,664 | Berechnet: {params['GESAMT']:,d}")
print(f"✅ Übereinstimmung: {params['GESAMT'] == 124337664}")

### 2.3 Self-Attention — Der Kern des Transformers

Self-Attention erlaubt jedem Token, mit allen anderen Token in der Sequenz zu interagieren.

**Formel:** $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

- **Q (Query):** „Wonach suche ich?"
- **K (Key):** „Was biete ich an?"
- **V (Value):** „Welche Information gebe ich weiter?"
- **$\sqrt{d_k}$:** Skalierung gegen zu große Dot-Produkte
- **Causal Mask:** Token darf nur auf sich selbst und vorherige Token schauen

In [ ]:
# Causal Attention Mask visualisieren
block_size = 16
mask = torch.tril(torch.ones(block_size, block_size))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Matrix-Darstellung
im = ax1.imshow(mask, cmap='Blues', aspect='auto')
ax1.set_title('Causal Attention Mask (1 = erlaubt, 0 = maskiert)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Key-Position (worauf geschaut wird)')
ax1.set_ylabel('Query-Position (wer schaut)')

# Erklärung
ax2.axis('off')
ax2.text(0.1, 0.9, '💡 Causal Mask erklärt:', fontsize=14, fontweight='bold', transform=ax2.transAxes)
ax2.text(0.1, 0.75, '• Token 0 darf nur Token 0 sehen', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.65, '• Token 5 darf Token 0-5 sehen', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.55, '• Token 15 darf alle 16 Token sehen', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.40, '• Verhindert "Cheating" — das Modell', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.35, '  kann nicht in die Zukunft schauen', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.20, '• Daher: autoregressive Generierung —', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.15, '  ein Token nach dem anderen', fontsize=12, transform=ax2.transAxes)

plt.tight_layout()
plt.show()

## 3. Daten vorbereiten

Wir trainieren auf Shakespeare — ein kleiner, lustiger Datensatz perfekt zum Experimentieren.

In [ ]:
# Shakespeare-Text laden
data_dir = '/opt/data/nanoGPT/data/shakespeare_char'
if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    
    # Shakespeare von Karpathys Repo herunterladen
    import urllib.request
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    text = urllib.request.urlopen(url).read().decode('utf-8')
    
    # Character-Level Tokenisierung
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}
    
    # Encode & als binär speichern
    data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    n = int(0.9 * len(data))
    train_data = data[:n]
    val_data = data[n:]
    
    train_data.numpy().astype(np.uint16).tofile(os.path.join(data_dir, 'train.bin'))
    val_data.numpy().astype(np.uint16).tofile(os.path.join(data_dir, 'val.bin'))
    
    # Metadaten speichern
    with open(os.path.join(data_dir, 'meta.pkl'), 'wb') as f:
        import pickle
        pickle.dump({'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}, f)
    
    print(f'✅ Shakespeare geladen: {len(text):,} Zeichen')
    print(f'✅ Vokabulargröße: {vocab_size} (Character-Level)')
    print(f'✅ Train: {len(train_data):,} | Val: {len(val_data):,}')
else:
    # Metadaten laden
    import pickle
    with open(os.path.join(data_dir, 'meta.pkl'), 'rb') as f:
        meta = pickle.load(f)
    vocab_size = meta['vocab_size']
    itos = meta['itos']
    stoi = meta['stoi']
    
    train_data = torch.from_numpy(np.fromfile(os.path.join(data_dir, 'train.bin'), dtype=np.uint16)).long()
    val_data = torch.from_numpy(np.fromfile(os.path.join(data_dir, 'val.bin'), dtype=np.uint16)).long()
    
    print(f'✅ Daten geladen: Train={len(train_data):,} | Val={len(val_data):,} | Vocab={vocab_size}')

In [ ]:
# Zeige Beispiel-Text und Tokenisierung
sample_text = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?"""

print('📝 Original-Text:')
print(sample_text[:200])
print()

# Tokenisieren
tokens = [stoi.get(c, 0) for c in sample_text]
print('🔢 Token-IDs (erste 50):')
print(tokens[:50])
print()

# Detokenisieren
decoded = ''.join(itos[t] for t in tokens)
print('🔄 Zurück-dekodiert:')
print(decoded[:200])
print(f'\n✅ Roundtrip korrekt: {sample_text == decoded}')

In [ ]:
# Batch-Visualisierung
def get_batch(split='train', batch_size=4, block_size=8):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch(batch_size=4, block_size=16)

fig, axes = plt.subplots(4, 1, figsize=(14, 8))
for i in range(4):
    input_text = ''.join(itos[t.item()] for t in x[i])
    target_text = ''.join(itos[t.item()] for t in y[i])
    
    # Hervorhebung: gleiche Zeichen grün, unterschiedliche rot
    colored = ''
    for inp, tgt in zip(input_text, target_text):
        if inp == tgt:
            colored += inp
        else:
            colored += f'[{inp}→{tgt}]'
    
    axes[i].text(0.5, 0.5, f'Batch {i+1}: {colored}',
                fontfamily='monospace', fontsize=11, ha='center', va='center',
                transform=axes[i].transAxes)
    axes[i].axis('off')

fig.suptitle('Trainingsdaten: Input → Target (nächstes Zeichen vorhersagen)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Modell initialisieren & Training

Wir trainieren ein kleines „Baby-GPT"-Modell — schnell genug für CPU, aber mit allen Komponenten eines echten GPTs.

In [ ]:
# Baby-GPT Konfiguration (schnell trainierbar, ~10M Parameter)
baby_config = GPTConfig(
    block_size=256,     # Kontextfenster: 256 Zeichen
    vocab_size=vocab_size,
    n_layer=6,          # 6 Transformer-Blöcke
    n_head=6,           # 6 Attention-Köpfe
    n_embd=384,         # Embedding-Dimension
    dropout=0.2,        # 20% Dropout (Regularisierung)
    bias=False,         # Kein Bias (wie GPT-2)
)

model = GPT(baby_config)
model.to(device)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'🏗️ Baby-GPT: {n_params:.1f}M Parameter')
print(f'   Block-Size: {baby_config.block_size}')
print(f'   Layers: {baby_config.n_layer}')
print(f'   Heads: {baby_config.n_head}')
print(f'   Embedding-Dim: {baby_config.n_embd}')
print(f'   Vocab-Size: {baby_config.vocab_size}')

In [ ]:
# Training-Hyperparameter
batch_size = 64
max_iters = 2000
eval_interval = 200
eval_iters = 100
learning_rate = 1e-3
min_lr = 1e-4
warmup_iters = 100
lr_decay_iters = max_iters

print(f'📊 Training-Konfiguration:')
print(f'   Batch-Size: {batch_size}')
print(f'   Max-Iterationen: {max_iters}')
print(f'   Learning-Rate: {learning_rate} → {min_lr} (Cosine Decay)')
print(f'   Warmup: {warmup_iters} Iterationen')
print(f'   Tokens pro Iteration: {batch_size * baby_config.block_size:,}')

In [ ]:
# Optimizer & Learning-Rate-Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.99),
    weight_decay=0.1,
)

def get_lr(it):
    """Cosine Decay mit linearem Warmup."""
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# LR-Schedule visualisieren
lrs = [get_lr(i) for i in range(max_iters)]
plt.plot(lrs)
plt.xlabel('Iteration')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule: Linear Warmup + Cosine Decay', fontweight='bold')
plt.axvline(warmup_iters, color='red', linestyle='--', alpha=0.5, label=f'Warmup ({warmup_iters})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 🚀 Training-Loop
@torch.no_grad()
def estimate_loss():
    """Schätzt Trainings- und Validierungs-Loss."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, baby_config.block_size)
            X, Y = X.to(device), Y.to(device)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

train_losses = []
val_losses = []
lrs_recorded = []

model.train()
t0 = time.time()

for iter_num in range(max_iters):
    # Learning Rate für diese Iteration
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    # Evaluation
    if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
        losses = estimate_loss()
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        lrs_recorded.append(lr)
        
        elapsed = time.time() - t0
        print(f'Iter {iter_num:5d}/{max_iters} | '
              f'Train-Loss: {losses["train"]:.4f} | '
              f'Val-Loss: {losses["val"]:.4f} | '
              f'LR: {lr:.2e} | '
              f'Zeit: {elapsed:.1f}s')
    
    # Trainings-Batch
    X, Y = get_batch('train', batch_size, baby_config.block_size)
    X, Y = X.to(device), Y.to(device)
    
    # Forward + Backward
    logits, loss = model(X, Y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - t0
print(f'\n✅ Training abgeschlossen in {total_time:.1f}s ({total_time/60:.1f}min)')

In [ ]:
# 📈 Loss-Kurven
eval_iters_list = list(range(0, max_iters, eval_interval)) + [max_iters - 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(eval_iters_list, train_losses, 'b-', label='Train Loss', linewidth=2)
ax1.plot(eval_iters_list, val_losses, 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Trainings-Verlauf', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Perplexity (e^loss)
train_ppl = [math.exp(l) for l in train_losses]
val_ppl = [math.exp(l) for l in val_losses]
ax2.plot(eval_iters_list, train_ppl, 'b-', label='Train Perplexity', linewidth=2)
ax2.plot(eval_iters_list, val_ppl, 'r-', label='Val Perplexity', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Perplexity (niedriger = besser)')
ax2.set_title('Perplexity = e^Loss', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'📊 Finale Metriken:')
print(f'   Train-Loss: {train_losses[-1]:.4f} | Perplexity: {math.exp(train_losses[-1]):.1f}')
print(f'   Val-Loss:   {val_losses[-1]:.4f} | Perplexity: {math.exp(val_losses[-1]):.1f}')

## 5. Text generieren

Jetzt der spannende Teil: Lass das trainierte Modell Text generieren!

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=0.8, top_k=None):
    """
    Generiert Text mit dem trainierten Modell.
    
    Parameter:
    - temperature: Kontrolliert die Kreativität (0 = deterministisch, 1 = kreativ, >1 = chaotisch)
    - top_k: Beschränkt Sampling auf die k wahrscheinlichsten Token
    """
    model.eval()
    
    # Prompt tokenisieren
    prompt_tokens = [stoi.get(c, 0) for c in prompt]
    idx = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
    
    for _ in range(max_new_tokens):
        # Kontext kürzen falls nötig
        idx_cond = idx if idx.size(1) <= baby_config.block_size else idx[:, -baby_config.block_size:]
        
        # Forward-Pass
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # Nur das letzte Token
        
        # Temperature anwenden
        logits = logits / temperature
        
        # Top-k Sampling
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        
        # Softmax & Sampling
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        
        # Anhängen
        idx = torch.cat((idx, idx_next), dim=1)
    
    # Dekodieren
    generated = ''.join(itos[t.item()] for t in idx[0])
    return generated

# Teste verschiedene Temperaturen
prompts = [
    "ROMEO:\n",
    "JULIET:\n",
    "First Citizen:\n",
]

for prompt in prompts:
    print(f'\n{"─"*70}')
    print(f'📝 Prompt: {repr(prompt)}')
    print(f'{"─"*70}')
    
    for temp in [0.5, 0.8, 1.2]:
        generated = generate(model, prompt, max_new_tokens=150, temperature=temp, top_k=40)
        print(f'\n🌡️ Temperature={temp}:')
        print(generated[:300])
        print('...' if len(generated) > 300 else '')

### 5.1 Temperature-Effekt visualisieren

Temperature kontrolliert die „Kreativität" des Modells:

In [ ]:
# Zeige, wie Temperature die Wahrscheinlichkeitsverteilung verändert
np.random.seed(42)
logits = torch.tensor([2.0, 1.0, 0.5, 0.1, -0.5, -1.0, -2.0])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
temperatures = [0.3, 0.7, 1.0, 2.0]

for ax, temp in zip(axes, temperatures):
    probs = F.softmax(logits / temp, dim=-1).numpy()
    bars = ax.bar(range(len(probs)), probs, color=plt.cm.viridis(probs / probs.max()))
    ax.set_title(f'Temperature = {temp}', fontweight='bold')
    ax.set_xlabel('Token-Index')
    ax.set_ylabel('Wahrscheinlichkeit')
    ax.set_ylim(0, 1)
    
    # Höchste Wahrscheinlichkeit markieren
    max_idx = probs.argmax()
    bars[max_idx].set_edgecolor('red')
    bars[max_idx].set_linewidth(3)

fig.suptitle('Temperature-Effekt auf die Wahrscheinlichkeitsverteilung',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Erkenntnis:')
print('   • T=0.3: Fast deterministisch — das wahrscheinlichste Token dominiert')
print('   • T=0.7: Ausgewogen — gute Balance aus Kreativität und Kohärenz')
print('   • T=1.0: Original-Verteilung — unverändert')
print('   • T=2.0: Abgeflacht — alle Token fast gleich wahrscheinlich (chaotisch)')

## 6. Attention visualisieren

Was „sieht" das Modell, wenn es Text generiert? Die Attention-Gewichte zeigen, welche Token miteinander interagieren.

In [ ]:
@torch.no_grad()
def get_attention_weights(model, text, layer_idx=0, head_idx=0):
    """Extrahiert Attention-Gewichte für eine bestimmte Schicht und einen Kopf."""
    model.eval()
    
    tokens = [stoi.get(c, 0) for c in text]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    
    # Forward-Pass mit output_attentions
    B, T = idx.shape
    
    # Token Embeddings
    tok_emb = model.transformer.wte(idx)
    pos_emb = model.transformer.wpe(torch.arange(0, T, dtype=torch.long, device=device))
    x = tok_emb + pos_emb
    x = model.transformer.drop(x)
    
    # Durch die Blöcke bis zur gewünschten Schicht
    for i, block in enumerate(model.transformer.h):
        if i == layer_idx:
            # Attention mit Gewichten
            ln_x = block.ln_1(x)
            B, T, C = ln_x.shape
            
            qkv = block.attn.c_attn(ln_x)
            q, k, v = qkv.split(C, dim=2)
            
            n_head = block.attn.n_head
            head_dim = C // n_head
            q = q.view(B, T, n_head, head_dim).transpose(1, 2)
            k = k.view(B, T, n_head, head_dim).transpose(1, 2)
            
            # Attention-Gewichte berechnen
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))
            att = att.masked_fill(
                torch.tril(torch.ones(T, T, device=device)).view(1, 1, T, T) == 0,
                float('-inf')
            )
            att = F.softmax(att, dim=-1)
            
            return att[0, head_idx].cpu().numpy()
        else:
            x = block(x)
    
    return None

# Beispiel-Text für Attention-Visualisierung
sample = "ROMEO:\nBut soft, what light through yonder window breaks?\nIt is the east, and Juliet is the sun."

# Attention aus verschiedenen Layern und Köpfen
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for layer in range(2):
    for head in range(3):
        ax = axes[layer, head]
        att_weights = get_attention_weights(model, sample, layer_idx=layer, head_idx=head)
        
        if att_weights is not None:
            im = ax.imshow(att_weights, cmap='YlOrRd', aspect='auto')
            ax.set_title(f'Layer {layer}, Head {head}', fontweight='bold')
            
            # Token-Labels (nur jedes 2. für Lesbarkeit)
            chars = list(sample[:att_weights.shape[0]])
            tick_labels = [c if i % 2 == 0 else '' for i, c in enumerate(chars)]
            ax.set_xticks(range(len(chars)))
            ax.set_xticklabels(tick_labels, fontsize=7, rotation=90)
            ax.set_yticks(range(len(chars)))
            ax.set_yticklabels(tick_labels, fontsize=7)

plt.colorbar(im, ax=axes, shrink=0.6, label='Attention-Gewicht')
fig.suptitle('🔍 Attention-Gewichte: Welche Token beachten einander?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Interpretation:')
print('   • Helle Zellen = starke Aufmerksamkeit zwischen zwei Token')
print('   • Frühe Layer: oft lokale Muster (benachbarte Zeichen)')
print('   • Späte Layer: semantische Beziehungen (Subjekt-Verb, Frage-Antwort)')
print('   • Verschiedene Köpfe spezialisieren sich auf verschiedene Muster')

## 7. Scaling Laws — Wie skaliert Performance?

Die Chinchilla-Scaling-Laws zeigen: Optimale Modelle haben ~20 Tokens pro Parameter.

Das bedeutet: Ein 1B-Parameter-Modell sollte auf ~20B Tokens trainiert werden.

In [ ]:
# FLOPs-Berechnung für Transformer (vereinfacht)
def estimate_flops(N_params, N_tokens):
    """
    Schätzt die FLOPs für das Training eines Transformers.
    Faustregel: ~6 FLOPs pro Parameter pro Token (Forward + Backward).
    """
    return 6 * N_params * N_tokens

def compute_optimal(N_flops):
    """
    Chinchilla-optimale Parameter- und Token-Anzahl für gegebenes FLOPs-Budget.
    N_opt ∝ C^0.5, D_opt ∝ C^0.5
    """
    # Chinchilla-Koeffizienten (aus dem Paper)
    N_opt = 0.089 * (N_flops ** 0.46)
    D_opt = 0.29 * (N_flops ** 0.53)
    return N_opt, D_opt

# Verschiedene FLOPs-Budgets
budgets = [1e15, 1e16, 1e17, 1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24]
labels = ['1 PF', '10 PF', '100 PF', '1 EF', '10 EF', '100 EF', '1 ZF', '10 ZF', '100 ZF', '1 YF']

print(f'{"FLOPs":>10s}  {"Optimale Params":>18s}  {"Optimale Tokens":>18s}  {"Tokens/Param":>14s}')
print('-' * 70)

for budget, label in zip(budgets, labels):
    N_opt, D_opt = compute_optimal(budget)
    ratio = D_opt / N_opt
    print(f'{label:>10s}  {N_opt/1e6:>10.1f}M        {D_opt/1e9:>10.1f}B        {ratio:>10.1f}')

print()
print('💡 Chinchilla-Regel: ~20 Tokens pro Parameter für optimale Compute-Effizienz')
print('   GPT-3 (175B): 300B Tokens → 1.7 Tokens/Param → UNTERtrainiert!')
print('   LLaMA (7B):   1T Tokens   → 143 Tokens/Param → ÜBERtrainiert (absichtlich für Inference)')

In [ ]:
# Visualisierung: Scaling Laws
flops_range = np.logspace(15, 24, 100)
N_opts, D_opts = compute_optimal(flops_range)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Parameter vs FLOPs
ax1.loglog(flops_range, N_opts, 'b-', linewidth=2, label='Optimale Parameter')
ax1.loglog(flops_range, D_opts, 'r--', linewidth=2, label='Optimale Tokens')
ax1.set_xlabel('Compute (FLOPs)')
ax1.set_ylabel('Anzahl')
ax1.set_title('Chinchilla Scaling Laws', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, which='both')

# Tokens/Parameter-Ratio
ratio = D_opts / N_opts
ax2.semilogx(flops_range, ratio, 'g-', linewidth=2)
ax2.axhline(20, color='red', linestyle='--', alpha=0.5, label='~20 Tokens/Param (Chinchilla)')
ax2.set_xlabel('Compute (FLOPs)')
ax2.set_ylabel('Tokens / Parameter')
ax2.set_title('Optimales Token-zu-Parameter-Verhältnis', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## 8. Modell speichern & laden

In [ ]:
# Modell speichern
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': baby_config,
    'iter_num': max_iters,
    'train_loss': train_losses[-1],
    'val_loss': val_losses[-1],
    'vocab_info': {'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi},
}

os.makedirs('/opt/data/nanoGPT/checkpoints', exist_ok=True)
torch.save(checkpoint, '/opt/data/nanoGPT/checkpoints/baby_gpt_shakespeare.pt')

file_size = os.path.getsize('/opt/data/nanoGPT/checkpoints/baby_gpt_shakespeare.pt') / 1e6
print(f'💾 Modell gespeichert: {file_size:.1f} MB')
print(f'   Pfad: /opt/data/nanoGPT/checkpoints/baby_gpt_shakespeare.pt')

In [ ]:
# Modell laden & weiter generieren
checkpoint = torch.load('/opt/data/nanoGPT/checkpoints/baby_gpt_shakespeare.pt', map_location=device)
loaded_model = GPT(checkpoint['config'])
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)
loaded_model.eval()

print(f'📂 Geladenes Modell: {sum(p.numel() for p in loaded_model.parameters())/1e6:.1f}M Parameter')
print(f'   Train-Loss: {checkpoint["train_loss"]:.4f}')
print(f'   Val-Loss:   {checkpoint["val_loss"]:.4f}')

# Generiere mit geladenem Modell
print('\n📝 Generierter Text (geladenes Modell):')
print('─' * 60)
generated = generate(loaded_model, 'KING HENRY:\n', max_new_tokens=200, temperature=0.8, top_k=40)
print(generated)

## 9. Zusammenfassung & Nächste Schritte

### ✅ Was du gelernt hast:
- **GPT-Architektur:** Token/Position-Embeddings → Transformer-Blöcke (Attention + MLP) → LM-Head
- **Self-Attention:** Q, K, V — jedes Token interagiert mit allen vorherigen Token
- **Causal Mask:** Verhindert „Blick in die Zukunft"
- **Training:** Cross-Entropy-Loss, AdamW, Cosine-Decay-LR-Schedule
- **Textgenerierung:** Autoregressiv, Temperature, Top-k-Sampling
- **Scaling Laws:** Chinchilla-optimal ≈ 20 Tokens/Parameter

### 🚀 Nächste Schritte:
1. **Größeres Modell trainieren** — GPT-2 Small (124M) auf OpenWebText
2. **Fine-Tuning** — Auf eigenen Daten (z.B. deutsche Texte) anpassen
3. **Byte-Pair Encoding (BPE)** — Statt Character-Level: Subword-Tokenisierung
4. **Flash Attention** — 2-4x schnelleres Training
5. **Mixture of Experts (MoE)** — Skalierung auf Billionen Parameter

### 📚 Referenzen:
- [nanoGPT (Karpathy)](https://github.com/karpathy/nanoGPT)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Chinchilla Scaling Laws](https://arxiv.org/abs/2203.15556)
- [Let's build GPT: from scratch (Karpathy-Video)](https://www.youtube.com/watch?v=kCc8FmEb1nY)